# Local MBS Character

Exploring local topological markers for Majorana Bound States (MBS) in standard Kitaev chain with varying hopping potential.

## Preamble

In [3]:
using Statistics
using LinearAlgebra
using Plots
using DataFrames
using BSON: @save, @load
using ProgressMeter
using Printf
using DataStructures: OrderedDict
using IterTools
using Random
using Measures
using Base.Threads: @threads, nthreads, threadid, SpinLock
using JLD2

In [4]:
base_dir = @__DIR__
results_dir = joinpath(base_dir, "PRB_test_results")
isdir(results_dir) || mkdir(results_dir)

true

In [5]:
# Global parameters

N_glob = 100
mu_glob = 0.0
Delta_glob = 0.5
t_n_glob = [1.0, 1.5]

mu_range_glob = collect(range(0.0, stop=3.0, length=31))
println("global param values set")

global param values set


## Model Construction & Diagonalisation

In [6]:
function create_bdg_hamiltonian(
    N::Int, 
    t_n::Vector{Float64}, 
    mu::Float64, 
    Delta::Float64, 
    sequence::Vector{Int}
)::Matrix{ComplexF64}
    H0 = zeros(ComplexF64, N, N)
    for i in 1:N
        H0[i, i] = -mu
    end
    for i in 1:N-1
        hopping_index = sequence[i]
        hopping_term = t_n[hopping_index]
        H0[i, i+1] = -hopping_term
        H0[i+1, i] = -hopping_term
    end

    Delta_matrix = zeros(ComplexF64, N, N)
    for i in 1:N-1
        Delta_matrix[i, i+1] = Delta
        Delta_matrix[i+1, i] = -Delta
    end

    BdG = zeros(ComplexF64, 2N, 2N)
    BdG[1:N, 1:N] = H0
    BdG[N+1:end, N+1:end] = -conj(H0)
    BdG[1:N, N+1:end] = Delta_matrix
    BdG[N+1:end, 1:N] = Delta_matrix'
    return BdG
end

create_bdg_hamiltonian (generic function with 1 method)

In [7]:
function solve_bdg_hamiltonian(
    N::Int,
    t_n::Vector{Float64},
    mu::Float64,
    Delta::Float64,
    sequence::Vector{Int}
)::Tuple{Vector{ComplexF64}, Matrix{ComplexF64}}

    BdG = create_bdg_hamiltonian(N, t_n, mu, Delta, sequence)
    F = eigen(BdG)
    return F.values, F.vectors
end

solve_bdg_hamiltonian (generic function with 1 method)

In [8]:
function log_range(base::Float64, start_power::Float64, end_power::Float64, length::Int)
    powers = range(start_power, stop=end_power, length=length)
    values = base .^ powers
    return values
end

function t_ranges_combs(t_ranges::Vector{Vector{Float64}})
    t_combinations_matrix = collect(IterTools.product(t_ranges...))
    t_combinations = [collect(t) for t in vec(t_combinations_matrix)]
    return t_combinations
end

t_ranges_combs (generic function with 1 method)

In [9]:
function initialise_df(
    N_range::Vector{Int},
    t_combinations::Vector{Vector{Float64}},
    mu_range::Vector{Float64},
    Delta_range::Vector{Float64},
    sequences::Vector{Vector{Int}};
    sequence_names::Union{Nothing, Vector{String}} = nothing,
    sequence_ids::Union{Nothing, Vector{Tuple{Float64, Float64}}} = nothing
)::DataFrame

    if sequence_names === nothing
        sequence_names = ["seq_$(i)" for i in 1:length(sequences)]
    end

    if sequence_ids === nothing
        sequence_ids = [(NaN, NaN) for _ in 1:length(sequences)]
    end
    @assert length(sequence_names) == length(sequences) "sequence_names must match sequences length"
    @assert length(sequence_ids) == length(sequences) "sequence_ids must match sequences length"
    
    df = DataFrame(
        N = Int[],
        t_n = Vector{Float64}[],
        mu = Float64[],
        Delta = Float64[],
        sequence_name = String[],
        sequence_id = Tuple{Float64, Float64}[],
        sequence = Vector{Int}[]
    )

    for N in N_range
        for t_n in t_combinations
            for mu in mu_range
                for Delta in Delta_range
                    for (seq, name, seq_id) in zip(sequences, sequence_names, sequence_ids)
                        push!(df, (N, t_n, mu, Delta, name, seq_id, seq))
                    end
                end
            end
        end
    end

    return df
end

initialise_df (generic function with 1 method)

In [10]:
function solve_for_all!(
    df::DataFrame
)::DataFrame

    n = nrow(df)
    vals_col = Vector{Vector{Float64}}(undef, n)
    vecs_col = Vector{Matrix{Float64}}(undef, n)

    @showprogress for (i, row) in enumerate(eachrow(df))
        vals, vecs = solve_bdg_hamiltonian(
            row.N,
            row.t_n,
            row.mu,
            row.Delta,
            row.sequence,
        )
        vals_col[i] = vals
        vecs_col[i] = vecs
    end

    df.eigenvalues = vals_col
    df.eigenvectors = vecs_col
    return df
end

function solve_for_all_no_prog!(
    df::DataFrame
)::DataFrame

    n = nrow(df)
    vals_col = Vector{Vector{Float64}}(undef, n)
    vecs_col = Vector{Matrix{Float64}}(undef, n)

    for (i, row) in enumerate(eachrow(df))
        vals, vecs = solve_bdg_hamiltonian(
            row.N,
            row.t_n,
            row.mu,
            row.Delta,
            row.sequence,
        )
        vals_col[i] = vals
        vecs_col[i] = vecs
    end

    df.eigenvalues = vals_col
    df.eigenvectors = vecs_col
    return df
end

solve_for_all_no_prog! (generic function with 1 method)

In [11]:
## Calculates rho for each row of the df and filter based on specified rho range, then solve for the filtered df

function solve_for_fixed_rho(
    df::DataFrame,
    rho_range::Vector{Float64}
)::DataFrame
    
    df.rho = [row.t_n[2] / row.t_n[1] for row in eachrow(df)]
    filtered_df = filter(row -> row.rho in rho_range, df)
    return solve_for_all!(filtered_df)
end

solve_for_fixed_rho (generic function with 1 method)

In [12]:
function solve_for_fixed_rho_Delta_ratio(
    df::DataFrame,
    rho_Delta_range::Vector{Float64},
)::DataFrame
    
    df.rho = [row.t_n[2] / row.t_n[1] for row in eachrow(df)]
    df.Delta = [row.Delta for row in eachrow(fixed_ratio_df)]
    df.rho_Delta = [row.rho / row.Delta for row in eachrow(df)]
    filtered_df = filter(row -> row.rho_Delta in rho_Delta_range, df)
    return solve_for_all!(filtered_df)
end

solve_for_fixed_rho_Delta_ratio (generic function with 1 method)

## Secondary Calculations

### Spectral Function

In [13]:
function calc_spectral_func_for_all!(
    df::DataFrame,
    eta::Float64,
    omega_range::Vector{Float64}
)::DataFrame

    n = nrow(df)
    spectral_funcs_col = Vector{Matrix{Float64}}(undef, n)
    
    # prog = Progress(n; desc="Spectral function", dt=1.0)
    # prog_lock = SpinLock()
    @showprogress for i in 1:n
        row = df[i, :]
        bdg_matrix = create_bdg_hamiltonian(row.N, row.t_n, row.mu, row.Delta, row.sequence)
        spectral_values = zeros(row.N, length(omega_range))
        
        for j in 1:length(omega_range)
            omega = omega_range[j]
            G = inv((omega+ im * eta) * I - bdg_matrix)
            
            for site_idx in 1:row.N
                particle_contribution = -imag(G[site_idx, site_idx]) / π
                hole_contribution = -imag(G[row.N + site_idx, row.N + site_idx]) / π
                spectral_values[site_idx, j] = (particle_contribution + hole_contribution) / 2
            end
        end
        spectral_funcs_col[i] = spectral_values
        
        # lock(prog_lock)
        # next!(prog)
        # unlock(prog_lock)
    end

    df[!, :spectral_function] = spectral_funcs_col
    return df
end

calc_spectral_func_for_all! (generic function with 1 method)

### Inverse Participation Ratio

### Global Majorana Polarisation

In [14]:
function np_calc_maj_mp(
    eigenvectors::Union{Matrix{Float64}, Matrix{ComplexF64}} # N.B. the eigen function gives only real eigenvectors since Hermitian is explicit
)::Float64
    num_rows = size(eigenvectors, 1)
    N = num_rows ÷ 2
    middle_state_index = N

    u = eigenvectors[1:N, middle_state_index]
    v = eigenvectors[(N+1):2N, middle_state_index]

    numerator_L = Float64(sum(u[i] * conj(v[i]) + v[i] * conj(u[i]) for i in 1:(N÷2)))
    denominator_L = Float64(sum(abs2(u[i]) + abs2(v[i]) for i in 1:(N÷2)))
    MP_L = numerator_L / denominator_L

    numerator_R = Float64(sum(u[i] * conj(v[i]) + v[i] * conj(u[i]) for i in ((N÷2)+1):N))
    denominator_R = Float64(sum(abs2(u[i]) + abs2(v[i]) for i in ((N÷2)+1):N))
    MP_R = numerator_R / denominator_R

    overall_MP = MP_L * MP_R

    return overall_MP
end

np_calc_maj_mp (generic function with 1 method)

In [15]:
function solve_mp_all!(
    df::DataFrame
)::DataFrame

    n = nrow(df)
    vals_col = Vector{Vector{Float64}}(undef, n)
    vecs_col = Vector{Matrix{Float64}}(undef, n)
    mp_col = Vector{Float64}(undef, n)

    @showprogress for (i, row) in enumerate(eachrow(df))
        vals, vecs = solve_bdg_hamiltonian(
            row.N,
            row.t_n,
            row.mu,
            row.Delta,
            row.sequence,
        )
        vals_col[i] = vals
        vecs_col[i] = vecs

        mp = np_calc_maj_mp(vecs)
        mp_col[i] = mp
    end

    df.eigenvalues = vals_col
    df.eigenvectors = vecs_col
    df.mp = mp_col
    
    return df
end

solve_mp_all! (generic function with 1 method)

### Local Majorana Polarisation

In [36]:
"""
    This function will take in a precalcualted results dataframe and
    calculate the local MP for the the middle state (candidate MBS) for
    each local chain across all local chain scales from x=1 to N/2.
"""

"""
    calc_local_scales_mp(eigenvectors::Matrix{Float64})

Given BdG eigenvectors (2N×2N), computes the local Majorana polarisation (MP) for all possible local chains at all even scales from 2 up to N.
Returns a vector of results, each entry is a NamedTuple with all numerators, denominators, MPs, and indices.

Each result contains:
    :scale         => scale (even integer, 2, 4, ..., N)
    :start_site    => starting site of the local chain (1-based)
    :end_site      => ending site of the local chain
    :numerator_L, :denominator_L, :MP_L
    :numerator_R, :denominator_R, :MP_R
    :overall_MP    => MP_L * MP_R
"""
function calc_local_scales_mp(
    eigenvectors::Matrix{Float64}
)::Vector{NamedTuple}
    
    num_rows = size(eigenvectors, 1)
    N = num_rows ÷ 2
    middle_state_index = N  # Use the middle eigenstate (candidate MBS)
    middle_plus_one_state_index = N + 1
    u_mid = eigenvectors[1:N, middle_state_index]
    v_mid = eigenvectors[N+1:end, middle_state_index]
    u_mpo = eigenvectors[1:N, middle_plus_one_state_index]
    v_mpo = eigenvectors[N+1:end, middle_plus_one_state_index]

    results = NamedTuple[]
    @showprogress for scale in 2:2:N # Only even scales to ensure equal left/right splits
        for start_site in 1:(N - scale + 1)
            end_site = start_site + scale - 1
            left_inds = start_site:(start_site + scale ÷ 2 - 1)
            right_inds = (start_site + scale ÷ 2):end_site

            numerator_L = sum(u_mid[i] * conj(v_mid[i]) + v_mid[i] * conj(u_mid[i]) for i in left_inds)
            denominator_L = sum(abs2(u_mid[i]) + abs2(v_mid[i]) for i in left_inds)
            MP_L = denominator_L ≈ 0 ? 0.0 : numerator_L / denominator_L

            numerator_R = sum(u_mpo[i] * conj(v_mpo[i]) + v_mpo[i] * conj(u_mpo[i]) for i in right_inds)
            denominator_R = sum(abs2(u_mpo[i]) + abs2(v_mpo[i]) for i in right_inds)
            MP_R = denominator_R ≈ 0 ? 0.0 : numerator_R / denominator_R

            overall_MP = MP_L * MP_R

            push!(results, (
                scale=scale,
                start_site=start_site,
                end_site=end_site,
                numerator_L=numerator_L,
                denominator_L=denominator_L,
                MP_L=MP_L,
                numerator_R=numerator_R,
                denominator_R=denominator_R,
                MP_R=MP_R,
                overall_MP=overall_MP
            ))
        end
    end
    return results
end

function calc_local_scales_mp_optimised(
    eigenvectors::Matrix{Float64}
)::Vector{NamedTuple}

    num_rows = size(eigenvectors, 1)
    N = num_rows ÷ 2
    middle_state_index = N  # Use the middle eigenstate (candidate MBS)
    u = eigenvectors[1:N, middle_state_index]
    v = eigenvectors[N+1:end, middle_state_index]

    # Pre-compute all (scale, start_site) pairs to flatten the workload
    tasks = NamedTuple{(:scale, :start_site), Tuple{Int, Int}}[]
    for scale in 2:2:N
        for start_site in 1:(N - scale + 1)
            push!(tasks, (scale=scale, start_site=start_site))
        end
    end

    # Pre-allocate results vector
    n_tasks = length(tasks)
    results = Vector{NamedTuple}(undef, n_tasks)

    # Parallel computation over all tasks
    @threads for i in 1:n_tasks
        scale, start_site = tasks[i]
        end_site = start_site + scale - 1
        left_inds = start_site:(start_site + scale ÷ 2 - 1)
        right_inds = (start_site + scale ÷ 2):end_site

        # Vectorised sums for slight speedup (Julia broadcasts efficiently)
        numerator_L = sum(u[left_inds] .* conj.(v[left_inds]) .+ v[left_inds] .* conj.(u[left_inds]))
        denominator_L = sum(abs2.(u[left_inds]) .+ abs2.(v[left_inds]))
        MP_L = denominator_L ≈ 0 ? 0.0 : numerator_L / denominator_L

        numerator_R = sum(u[right_inds] .* conj.(v[right_inds]) .+ v[right_inds] .* conj.(u[right_inds]))
        denominator_R = sum(abs2.(u[right_inds]) .+ abs2.(v[right_inds]))
        MP_R = denominator_R ≈ 0 ? 0.0 : numerator_R / denominator_R

        overall_MP = MP_L * MP_R

        results[i] = (
            scale=scale,
            start_site=start_site,
            end_site=end_site,
            numerator_L=numerator_L,
            denominator_L=denominator_L,
            MP_L=MP_L,
            numerator_R=numerator_R,
            denominator_R=denominator_R,
            MP_R=MP_R,
            overall_MP=overall_MP
        )
    end

    return results
end

function calc_local_scales_mp_allscales_optimised(
    eigenvectors::Union{Matrix{Float64}, Matrix{BigFloat}, Matrix{Complex{Float64}}, Matrix{Complex{BigFloat}}}
)::Vector{NamedTuple}

    num_rows = size(eigenvectors, 1)
    N = num_rows ÷ 2
    middle_state_index = N
    middle_plus_one_state_index = N + 1
    u_mid = eigenvectors[1:N, middle_state_index]
    v_mid = eigenvectors[N+1:end, middle_state_index]
    u_mpo = eigenvectors[1:N, middle_plus_one_state_index]
    v_mpo = eigenvectors[N+1:end, middle_plus_one_state_index]

    # Pre-compute all (scale, start_site) pairs for all scales
    tasks = NamedTuple{(:scale, :start_site), Tuple{Int, Int}}[]
    for scale in 1:N
        for start_site in 1:(N - scale + 1)
            push!(tasks, (scale=scale, start_site=start_site))
        end
    end

    n_tasks = length(tasks)
    num_threads = Threads.nthreads()
    results = Vector{NamedTuple}(undef, n_tasks)

    Threads.@threads for thread_idx in 1:num_threads
        for i in thread_idx:num_threads:n_tasks
            scale, start_site = tasks[i]
            end_site = start_site + scale - 1

            # For odd scales, include the middle site in both left and right
            if isodd(scale)
                mid = start_site + (scale - 1) ÷ 2
                left_inds = start_site:mid
                right_inds = mid:end_site
            else
                left_inds = start_site:(start_site + scale ÷ 2 - 1)
                right_inds = (start_site + scale ÷ 2):end_site
            end

            numerator_L = sum(u_mid[left_inds] .* conj.(v_mid[left_inds]) .+ v_mid[left_inds] .* conj.(u_mid[left_inds]))
            denominator_L = sum(abs2.(u_mid[left_inds]) .+ abs2.(v_mid[left_inds]))
            MP_L = numerator_L / denominator_L

            numerator_R = sum(u_mpo[right_inds] .* conj.(v_mpo[right_inds]) .+ v_mpo[right_inds] .* conj.(u_mpo[right_inds]))
            denominator_R = sum(abs2.(u_mpo[right_inds]) .+ abs2.(v_mpo[right_inds]))
            MP_R = numerator_R / denominator_R

            overall_MP = conj(MP_L) * MP_R
            try
                overall_MP = Float64(overall_MP)
            catch err
                @warn "overall_MP is complex, taking real part. Details:" value=overall_MP error=err
                overall_MP = Float64(real(overall_MP))
            end

            results[i] = (
                scale=scale,
                start_site=start_site,
                end_site=end_site,
                numerator_L=numerator_L,
                denominator_L=denominator_L,
                MP_L=MP_L,
                numerator_R=numerator_R,
                denominator_R=denominator_R,
                MP_R=MP_R,
                overall_MP=overall_MP
            )
        end
    end

    return results
end


calc_local_scales_mp_allscales_optimised (generic function with 1 method)

## Hoping Sequence Generation

### Mechanical Inflation Sequences

In [17]:
function normal_crystal_gen(N::Int)
    return fill(1, N)
end

function golden_ratio_sequence_gen(N::Int)
    sequence = "A"
    while length(sequence) < N
        sequence = replace(sequence, "A" => "AB", "B" => "A")
    end
    # println(sequence)
    
    number_sequence = [ch == 'A' ? 1 : 2 for ch in sequence]
    
    return number_sequence
end

function silver_ratio_sequence_gen(N::Int)
    sequence = "A"
    while length(sequence) < N
        sequence = replace(sequence, "A" => "BAA", "B" => "A")
    end
    # println(sequence)
    
    number_sequence = [ch == 'A' ? 1 : 2 for ch in sequence]
    
    return number_sequence
end

function check_PV_ratio(number_sequence::Vector{Int})
    count_A = count(x -> x == 1, number_sequence)
    count_B = count(x -> x == 2, number_sequence)
    
    ratio = count_A / count_B
    
    return ratio
end

function thue_morse_sequence_gen(n::Int)
    if n <= 0 || (n & (n - 1)) != 0
        throw(ArgumentError("n must be a power of 2."))
    end
    
    sequence = Int[]
    for i in 0:n-1
        num_ones = count(c -> c == '1', string(i, base=2))
        push!(sequence, num_ones % 2 + 1)
    end
    
    return sequence
end

function plastic_ratio_sequence_gen(N::Int)
    sequence = "A"
    while length(sequence) < N
        sequence = replace(sequence, "A" => "B", "B" => "AC", "C" => "A") # Van der Laan word generation
        # sequence = replace(sequence, "A" => "AB", "B" => "AC", "C" => "A") # ChatGPT
    end
    # println(sequence)
    
    number_sequence = [ch == 'A' ? 1 : ch == 'B' ? 2 : 3 for ch in sequence]
    
    return number_sequence
end

function check_plastic_ratio(number_sequence::Vector{Int}, rho::Float64)
    count_A = count(x -> x == 1, number_sequence)
    count_B = count(x -> x == 2, number_sequence)
    count_C = count(x -> x == 3, number_sequence)

    ratio_AB = count_A / count_B
    ratio_BC = count_B / count_C

    return (ratio_AB, ratio_BC, rho)
end

function tribonacci_SeqGen(N::Int)
    sequence = "A"
    while length(sequence) < N
        sequence = replace(sequence, "A" => "AB", "B" => "AC", "C" => "A")
    end
    
    number_sequence = [ch == 'A' ? 1 : ch == 'B' ? 2 : 3 for ch in sequence]

    return number_sequence
end


# Create pairing parameter sequence for N sites
N_seed = 4096


# Normal Crystal
N_normal = N_seed
normal_sequence = normal_crystal_gen(N_normal)

# Golden Ratio
N_gold = N_seed
golden_sequence = golden_ratio_sequence_gen(N_gold)

# Silver Ratio
N_silver = N_seed
silver_sequence = silver_ratio_sequence_gen(N_silver)

# Thue-Morse 
N_thue_morse = N_seed
thue_morse_sequence = thue_morse_sequence_gen(N_thue_morse)

# Plastic Ratio
N_plastic = N_seed
plastic_sequence = plastic_ratio_sequence_gen(N_plastic)

println("Sequences generated.")

Sequences generated.


### Continued Fraction Slopes Cut-and-Project

### Minimal Rational Approx Cut-and-Project

In [18]:
## Generate full set of slopes

N = 1000
map_to = (2,1)
qmin, qmax = 2, N

# 1. Threaded generation of all reduced rationals p/q in [0,1]
slopes = Float64[]
slope_lock = SpinLock()
slope_prog = Progress(qmax - qmin + 1; desc="Generating slopes")

@threads for q in qmin:qmax
    local_chunk = Float64[]
    sizehint!(local_chunk, q)
    for p in 1:q-1
        if gcd(p, q) == 1
            push!(local_chunk, p / q)
        end
    end
    lock(slope_lock)
    append!(slopes, local_chunk)
    next!(slope_prog)
    unlock(slope_lock)
end

push!(slopes, 0.0, 1.0)
sort!(slopes)
unique!(slopes)

println("Initially generated $(length(slopes)) slopes.")
# println("Last 10 slopes: ", slopes[end-10:end])

Generating slopes 100%|██████████████████████████████████| Time: 0:00:00


Initially generated 304193 slopes.


In [19]:
## Optionally rebalance slopes to fixed density

nbins = 100
nperbin = 1
edges = range(0.0, stop=1.0, length=nbins + 1)

bin_idx = clamp.(searchsortedfirst.(Ref(edges), slopes) .- 1, 1, nbins)
selected_unique = falses(length(slopes))
counts = zeros(Int, nbins)
for i in eachindex(slopes)
    b = bin_idx[i]
    if counts[b] < nperbin
        selected_unique[i] = true
        counts[b] += 1
    end
end

slopes = slopes[selected_unique]
println("Rebalanced to $(length(slopes)) unique slopes (nbins=$nbins, nperbin=$nperbin).")
# println("Last 10 slopes: ", slopes[end-10:end])

Rebalanced to 100 unique slopes (nbins=100, nperbin=1).


In [20]:
# ## Optionally filter for target slope or override to set custom slopes


# ## Optional: Filter for target slope
# target_slope = 2/(1 + sqrt(5)) # Set to a Float64 (e.g. 1/phi) to enable filtering, or nothing to disable
# slope_tol = 0.001

# if !isnothing(target_slope)
#     filter!(s -> abs(s - target_slope) <= slope_tol, slopes)
#     println("Filtered to $(length(slopes)) slopes within tolerance $slope_tol of target $target_slope.")
#     if isempty(slopes)
#         @warn "No slopes remained after filtering for target $target_slope with tolerance $slope_tol."
#     end
# end

## Overwrite to define literal slope
slopes = [0.6180048661800487]
# slopes = [target_slope]


1-element Vector{Float64}:
 0.6180048661800487

In [21]:
function cut_and_project_sequence(N::Int, slope::Float64; phason::Float64=0.0, map_to::Tuple{Int,Int}=(1,2))
    @assert N >= 1 "N must be >= 1"
    # @assert 0.0 < slope < 1.0 "slope expected in (0,1) for standard Sturmian words"
    phason = phason - floor(phason) # wrap into [0,1)
    seq = Vector{Int}(undef, N)
    frac = phason
    for n in 1:N
        frac += slope
        if frac >= 1.0
            frac -= 1.0
            # crossing an integer -> difference = 1
            seq[n] = map_to[1]   # "A"
        else
            seq[n] = map_to[2]   # "B"
        end
    end
    return seq
end

cut_and_project_sequence (generic function with 1 method)

In [22]:
phason_angle_range = [0.0] #collect(0.0:0.01:1.0)
n_slopes = length(slopes)
n_phason = length(phason_angle_range)
slot_count = n_slopes * n_phason

seqs = Vector{Vector{Int}}(undef, slot_count)
phis = Vector{Float64}(undef, slot_count)
phasons = Vector{Float64}(undef, slot_count)

seq_lock = SpinLock()
seq_prog = Progress(n_phason; desc="Cut-and-project")

@threads for ip in 1:n_phason
    phason = phason_angle_range[ip]
    base = (ip - 1) * n_slopes
    for is in 1:n_slopes
        idx = base + is
        phi = slopes[is]
        seqs[idx] = cut_and_project_sequence(N, phi; phason=phason, map_to=map_to)
        phis[idx] = phi
        phasons[idx] = phason
    end
    lock(seq_lock); next!(seq_prog); unlock(seq_lock)
end
println("Generated $(length(phis)) sequences across $(length(phason_angle_range)) phasons.")
# println("Sample Example Sequence (phi=$(phis[400]), phason=$(phasons[400])): ", seqs[400][1:50])


Generated 1 sequences across 1 phasons.


## Plotting

### Eigenvalues vs $\mu$ (or general x-axis)

In [23]:
function plt_eigenvalues_generalised(df::DataFrame, x_variable::Symbol, savepath::String; fixed_values...)
    """
    Plots eigenvalues as a function of a specified variable from the DataFrame.

    Parameters:
    df           -- DataFrame containing the data.
    x_variable   -- Symbol representing the variable to plot on the x-axis.
    fixed_values -- Keyword arguments specifying fixed values for all other DataFrame columns.
    """

    plt = Plots.scatter(
        legend=false, 
        xlabel=string(x_variable), 
        ylabel="E", 
        grid=true,
        size=(800,600),
        # ylims=(-0.1,0.1)
    )

    filtered_df = filter(row -> all(getproperty(row, key) == value for (key, value) in fixed_values), df)

    if isempty(filtered_df)
        error("No results match the specified fixed values.")
    end

    x_values = filtered_df[:, x_variable]
    all_eigenvalues = filtered_df[:, :eigenvalues]

    num_eigenvalues = length(all_eigenvalues[1])

    for i in 1:num_eigenvalues
        eigenvalue_line = [eigenvalues[i] for eigenvalues in all_eigenvalues]
        plot!(x_values, eigenvalue_line, color=:steelblue, alpha=0.8)
    end

    Plots.savefig(savepath)
    # Plots.display(plt)
end

plt_eigenvalues_generalised (generic function with 1 method)

In [24]:
function plt_eigenvalues_with_mp(df::DataFrame, x_variable::Symbol, savepath::String; fixed_values...)
    """
    Plots eigenvalues as a function of a specified variable from the DataFrame, overlaid with the global Majorana Polarisation (MP).

    Parameters:
    df           -- DataFrame containing the data (must include :mp column).
    x_variable   -- Symbol representing the variable to plot on the x-axis.
    savepath     -- String path to save the plot.
    fixed_values -- Keyword arguments specifying fixed values for all other DataFrame columns.
    """

    filtered_df = filter(row -> all(getproperty(row, key) == value for (key, value) in fixed_values), df)

    if isempty(filtered_df)
        error("No results match the specified fixed values.")
    end

    x_values = filtered_df[:, x_variable]
    all_eigenvalues = filtered_df[:, :eigenvalues]
    mp_values = filtered_df[:, :mp]

    num_eigenvalues = length(all_eigenvalues[1])

    # Create plot with primary y-axis for eigenvalues
    plt = Plots.plot(
        legend=:topright, 
        xlabel=string(x_variable), 
        ylabel="E", 
        grid=true,
        size=(800,600),
        # ylims=(-0.1,0.1)  # Uncomment if needed
    )

    # Plot eigenvalues
    for i in 1:num_eigenvalues
        eigenvalue_line = [eigenvalues[i] for eigenvalues in all_eigenvalues]
        plot!(x_values, eigenvalue_line, color=:steelblue, alpha=0.8, label=i == 1 ? "Eigenvalues" : "")
    end

    # Add MP on secondary y-axis
    plot!(twinx(), x_values, mp_values, 
          color=:red, linewidth=2, label="Global MP", 
          ylabel="MP", ylims=(-1.1, 1.1))

    Plots.savefig(savepath)
end

plt_eigenvalues_with_mp (generic function with 1 method)

### Local $\psi_\text{MBS}$ Amplitude

In [25]:
function calculate_probability_distributions_bdg(eigenvectors::Matrix{Float64})
    """
    Calculates the probability distributions of site occupancy from the eigenvectors using the BdG formalism.

    Returns:
    probabilities -- 2D array where each row corresponds to a site and each column corresponds to an energy level
    """
    N = size(eigenvectors, 1) ÷ 2  # Number of sites
    probabilities = zeros(Float64, N, size(eigenvectors, 2))

    for i in 1:size(eigenvectors, 2)  # Iterate over energy levels
        for n in 1:N  # Iterate over sites
            u_n_i = eigenvectors[n, i]  # Particle component
            v_n_i = eigenvectors[n+N, i]  # Hole component
            probabilities[n, i] = 0.5 * (abs(u_n_i)^2 + abs(v_n_i)^2)
        end
    end
    return probabilities
end

function majorana_linear_combinations(eigenvectors::Matrix{Float64})
    N = size(eigenvectors, 1) ÷ 2
    # Middle two eigenstates (columns)
    psi1 = eigenvectors[:, N]
    psi2 = eigenvectors[:, N+1]
    # Linear combinations
    left_majorana  = (psi1 + psi2) / sqrt(2)
    right_majorana = (psi1 - psi2) / sqrt(2)
    return left_majorana, right_majorana
end

function site_probabilities(psi::Vector{Float64})
    N = length(psi) ÷ 2
    u = psi[1:N]
    v = psi[N+1:end]
    return 0.5 .* (abs2.(u) .+ abs2.(v))
end

function plt_local_mbs_wavefunction(df::DataFrame, savepath::String; fixed_values...)
    """
    Plots the local wavefunction of the middle and middle+1 eigenstates as a function of site index.

    Parameters:
    df           -- DataFrame containing the data.
    savepath     -- String path to save the plot.
    fixed_values -- Keyword arguments specifying fixed values for all other DataFrame columns.
    """

    plt = Plots.scatter(
        legend=true, 
        xlabel="Site Index", 
        ylabel="Wavefunction Amplitude", 
        grid=true,
        size=(800,600),
        title="Local Wavefunction of MBS Eigenstates µ=$(fixed_values[:mu])"
    )

    filtered_df = filter(row -> all(getproperty(row, key) == value for (key, value) in fixed_values), df)

    if isempty(filtered_df)
        error("No results match the specified fixed values.")
    end

    N = size(filtered_df[!, :eigenvectors][1], 1) ÷ 2

    # (1)
    # uses calculate_probabilities_bdg to get the probability distributions for all eigenstates, then extracts the middle and middle+1 states
    probabilities = calculate_probability_distributions_bdg(filtered_df[!, :eigenvectors][1])
    middle_state_wavefunction = probabilities[:, N]  # Middle state (N-th eigenstate)
    middle_plus_one_state_wavefunction = probabilities[:, N+1]  # Middle+1 state (N+1-th eigenstate)
    x_values = 1:N

    plot!(x_values, abs.(middle_state_wavefunction), color=:forestgreen, alpha=0.8)
    plot!(x_values, abs.(middle_plus_one_state_wavefunction), color=:coral, alpha=0.8) 



    # # # (2)
    # # # uses linear combinations
    # eigenvectors = filtered_df[!, :eigenvectors][1]
    # left_maj, right_maj = majorana_linear_combinations(eigenvectors)
    # left_probs = site_probabilities(left_maj)
    # right_probs = site_probabilities(right_maj)
    # x_values = 1:length(left_probs)
    
    # plot!(x_values, abs.(left_probs), color=:forestgreen, alpha=0.8, label="psi1 + psi2")
    # plot!(x_values, abs.(right_probs), color=:coral, alpha=0.8, label="psi1 - psi2")

    # # print mu value in annotation
    # mu_value = filtered_df[!, :mu][1]
    # annotation_text = @sprintf("μ = %.2f", mu_value)
    # annotate!(0.5, 0.95, text(annotation_text, :left, 12, "black"), xycoords=:relative)

    Plots.savefig(savepath)
end

plt_local_mbs_wavefunction (generic function with 1 method)

### Spectral Function

In [26]:
function plt_spectral_function(df::DataFrame, omega_range::Vector{Float64}, savepath::String; fixed_values...)
    """
    Plots the spectral function as a function of energy for each site.

    Parameters:
    df           -- DataFrame containing the data (must include :spectral_function column).
    omega_range  -- Vector of energy values for the spectral function.
    savepath     -- String path to save the plot.
    fixed_values -- Keyword arguments specifying fixed values for all other DataFrame columns.
    """
    filtered_df = filter(row -> all(getproperty(row, key) == value for (key, value) in fixed_values), df)
    if isempty(filtered_df)
        error("No results match the specified fixed values.")
    end

    if hasproperty(filtered_df, :mp)
        mp_value = filtered_df[!, :mp][1]
        title_str = @sprintf("Spectral Function A(ω) for μ=%.3f, MP=%.2f", fixed_values[:mu], mp_value)
    else
        title_str = @sprintf("Spectral Function A(ω) for μ=%.3f", fixed_values[:mu])
    end

    spectral_function = filtered_df[!, :spectral_function][1]  # Assuming one matching row
    site_indices = 1:size(spectral_function, 1)

    heatmap(omega_range, site_indices, spectral_function,
            xlabel="ω",
            ylabel="Site Index",
            title=title_str,
            color=:viridis,
            clims = (0, maximum(spectral_function)),
            colorbar=true
    )

    Plots.savefig(savepath)
end

plt_spectral_function (generic function with 1 method)

### Local MP at scales

In [50]:
function plt_local_mp_scales_first_site(df::DataFrame, savepath::String; fixed_values...)
    """
    For each scale, plots the local MP for all local chains of that scale.
    x axis is the global chain, y axis is an arbitrary index for the local chain (start site),
    colour is the local MP value.

    Parameters:
    df           -- DataFrame containing the data.
    savepath     -- String path to save the plot.
    fixed_values -- Keyword arguments specifying fixed values for all other DataFrame columns.
    """
    filtered_df = filter(row -> all(getproperty(row, key) == value for (key, value) in fixed_values), df)
    if isempty(filtered_df)
        error("No results match the specified fixed values.")
    end
    
    # Extract the precalculated local MP array for the matching row
    local_mp_data = filtered_df[!, :local_mp][1] 

    # Extract axes and colour data
    scales = [data.scale for data in local_mp_data]
    start_sites = [data.start_site for data in local_mp_data]
    overall_mps = [data.overall_MP for data in local_mp_data]

    # Fix clims to -1,0
    clims = (-1.0, 0.0)

    # Create the scatter plot acting as a triangular heatmap
    plt = Plots.scatter(
        start_sites, 
        scales, 
        zcolor=overall_mps,
        color=:viridis,
        markershape=:square,
        markersize=4,
        markerstrokewidth=0,
        xlabel="Start Site",
        ylabel="Scale (Subchain Length)",
        title="Local MP across Scales\n$(join(["$k=$v" for (k,v) in fixed_values], ", "))",
        colorbar_title="Overall MP",
        size=(800, 600),
        grid=false,
        clims=clims,
        framestyle=:box
    )

    Plots.savefig(plt, savepath)
end

function plt_local_mp_scales_middle_site(df::DataFrame, savepath::String; log::Bool=false, fixed_values...)
    """
    For each scale, plots the local MP for all local chains of that scale.
    x axis is the middle site of the local chain, y axis is the scale, colour is the local MP value.
    """
    filtered_df = filter(row -> all(getproperty(row, key) == value for (key, value) in fixed_values), df)
    # if isempty(filtered_df)
    #     error("No results match the specified fixed values.")
    # end

    isempty(filtered_df) && error("No results match fixed_values=$(fixed_values)")
    nrow(filtered_df) == 1 || error("Expected exactly 1 row, got $(nrow(filtered_df)); fixed_values=$(fixed_values)")

    local_mp_data = filtered_df[!, :local_mp][1]

    scales = [data.scale for data in local_mp_data]
    middle_sites = [(data.start_site + data.end_site) ÷ 2 for data in local_mp_data]
    overall_mps = [data.overall_MP for data in local_mp_data]

    clims = (minimum(overall_mps), maximum(overall_mps)) #(-1.0, 0.0)

    # println("scales: ", scales)
    # println("middle_sites: ", middle_sites)
    # println("overall_mps: ", overall_mps)
    # println("clims: ", clims)
    # println("lengths: ", (length(scales), length(middle_sites), length(overall_mps)))

    colour_label_str = log ? "log10(|Overall MP|)" : "Overall MP"
    if log
        overall_mps = log10.(abs.(overall_mps))
        clims = (minimum(overall_mps), maximum(overall_mps))
    end

    plt = Plots.scatter(
        middle_sites,
        scales,
        zcolor=overall_mps,
        color=:viridis,
        markershape=:square,
        markersize=4,
        markerstrokewidth=0,
        xlabel="Middle Site",
        ylabel="Scale (Subchain Length)",
        title="Local MP across Scales\n$(join(["$k=$v" for (k,v) in fixed_values], ", "))",
        colorbar_title=colour_label_str,
        size=(800, 600),
        grid=false,
        clims=clims,
        framestyle=:box
    )

    Plots.savefig(plt, savepath)
end

plt_local_mp_scales_middle_site (generic function with 1 method)

## Executables

### 1.0 Sample slopes over $\mu$ range

In [62]:
## Solve for all slopes over range of mu at fixed rho and Delta
## Initialise parameter dataframe 

N_range = [90]
mu_range = mu_range = [0.01] #collect(range(0.0, stop=3.0, length=101)) #[0.0, 0.2, 1.0, 1.45]
Delta_range = [0.01] #collect(range(0.0, 1.0, 101))
t1_range = collect(range(1.0,1.0,1))
t2_range = collect(range(0.0,0.01,6))
t_ranges = [t1_range, t2_range]
t_combinations = t_ranges_combs(t_ranges)
sequences = seqs
sequence_ids = [(phis[i], phasons[i]) for i in 1:length(seqs)]

param_df = initialise_df(
    N_range,
    t_combinations,
    mu_range,
    Delta_range,
    seqs,
    sequence_ids = sequence_ids
)

println("Parameter DataFrame initialised with $(nrow(param_df)) rows.")

Parameter DataFrame initialised with 6 rows.


In [63]:
## Generate unique path

slopes_extracted = [seq_id[1] for seq_id in sequence_ids]
phasons_extracted = [seq_id[2] for seq_id in sequence_ids]

unique_path = @sprintf("N(%d-%d-%d)_mu(%.3f-%.3f-%d)_t1%.3f-%.3f-%d_t2%.3f-%.3f-%d_Delta%.3f-%.3f-%d_slope(%.3f-%.3f-%d)_phason(%.3f-%.3f-%d)",
    N_range[1], N_range[end], length(N_range),
    mu_range[1], mu_range[end], length(mu_range),
    t1_range[1], t1_range[end], length(t1_range),
    t2_range[1], t2_range[end], length(t2_range),
    Delta_range[1], Delta_range[end], length(Delta_range),
    minimum(slopes_extracted), maximum(slopes_extracted), length(unique(slopes_extracted)),
    minimum(phasons_extracted), maximum(phasons_extracted), length(unique(phasons_extracted))
)

"N(90-90-1)_mu(0.010-0.010-1)_t11.000-1.000-1_t20.000-0.010-6_Delta0.010-0.010-1_slope(0.618-0.618-1)_phason(0.000-0.000-1)"

In [64]:
## Solve 

df = solve_for_all!(param_df)
println("DataFrame with eigenvalues computed:")

DataFrame with eigenvalues computed:


In [ ]:
# Save df to .jld2 file
jld2_path = joinpath(results_dir, "data_$unique_path.jld2")

jldsave(jld2_path; df)
println("Saved DataFrame to JLD2 file at: $jld2_path")

Saved DataFrame to JLD2 file at: /Users/Will/Documents/Quasicrystal_Majorana_project_clone/Quasicrystal_Majorana_project/working_notebooks/PRB_test_results/data_N(200-200-1)_mu(0.000-3.000-101)_t11.000-1.000-1_t20.000-1.000-101_Delta0.050-0.050-1_slope(0.618-0.618-1)_phason(0.000-0.000-1).jld2


In [ ]:
# jld2_path = "/Users/Will/Documents/Quasicrystal_Majorana_project_clone/Quasicrystal_Majorana_project/working_notebooks/PRB_test_results/data_N(200-200-1)_mu(-4.000-4.000-401)_t11.000-1.000-1_t20.000-1.000-11_Delta0.050-0.050-1_slope(0.618-0.618-1)_phason(0.000-0.000-1).jld2"

# data = JLD2.load(jld2_path)
# df = data["df"]  # Replace "df" with the actual key if different
# println("Loaded DataFrame from JLD2 file. Number of rows: ", nrow(df), ", Number of columns: ", ncol(df))

# # extract unique path from new dataframe, must first extract parameters from df and then reformat into unique path string
# slopes_extracted = [seq_id[1] for seq_id in df.sequence_id]
# phasons_extracted = [seq_id[2] for seq_id in df.sequence_id]
# N_range_extracted = unique(df.N)
# mu_range_extracted = unique(df.mu)
# Delta_range_extracted = unique(df.Delta)
# t1_range_extracted = unique([t[1] for t in df.t_n])
# t2_range_extracted = unique([t[2] for t in df.t_n])

# unique_path = @sprintf("N(%d-%d-%d)_mu(%.3f-%.3f-%d)_t1%.3f-%.3f-%d_t2%.3f-%.3f-%d_Delta%.3f-%.3f-%d_slope(%.3f-%.3f-%d)_phason(%.3f-%.3f-%d)",
#     N_range_extracted[1], N_range_extracted[end], length(N_range_extracted),
#     mu_range_extracted[1], mu_range_extracted[end], length(mu_range_extracted),
#     t1_range_extracted[1], t1_range_extracted[end], length(t1_range_extracted),
#     t2_range_extracted[1], t2_range_extracted[end], length(t2_range_extracted),
#     Delta_range_extracted[1], Delta_range_extracted[end], length(Delta_range_extracted),
#     minimum(slopes_extracted), maximum(slopes_extracted), length(unique(slopes_extracted)),
#     minimum(phasons_extracted), maximum(phasons_extracted), length(unique(phasons_extracted))
# )

# t_combinations = t_ranges_combs([t1_range_extracted, t2_range_extracted])

In [ ]:
## Make eigenvalue vs. mu plots

dir = joinpath(results_dir, "eigs_vs_mu/$unique_path/svg")
isdir(dir) || mkpath(dir)

seen_t_ns = Set{Vector{Float64}}()
seen_seq_ids = Set{Tuple{Float64, Float64}}()

for t_n in t_combinations
    push!(seen_t_ns, t_n)
    for seq_id in sequence_ids
        push!(seen_seq_ids, seq_id)
        plt_eigenvalues_generalised(
            df,
            :mu,
            joinpath(dir, "eigenvalues_vs_mu_N$(N_range[1])_Delta$(Delta_range[1])_t1$(t_n[1])_t2$(t_n[2])_seq$(seq_id).svg");
            sequence_id=seq_id,
            t_n=t_n
        )
    end
end

println(length(seen_t_ns), " unique t_n combinations plotted.")
println(length(seen_seq_ids), " unique sequence_ids plotted.")

51 unique t_n combinations plotted.
1 unique sequence_ids plotted.


In [ ]:
## Make eigs vs. mu + global MP plot

# calculate global MP for all rows
df = solve_mp_all!(df)

# plot eigenvalues vs. mu with MP colouring
dir = joinpath(results_dir, "eigs_vs_mu_with_mp/$unique_path")
isdir(dir) || mkpath(dir)

seen_t_ns = Set{Vector{Float64}}()
seen_seq_ids = Set{Tuple{Float64, Float64}}()

for t_n in t_combinations
    push!(seen_t_ns, t_n)
    for seq_id in sequence_ids
        push!(seen_seq_ids, seq_id)
        plt_eigenvalues_with_mp(
            df,
            :mu,
            joinpath(dir, "eigenvalues_vs_mu_with_mp_N$(N_range[1])_Delta$(Delta_range[1])_t1$(t_n[1])_t2$(t_n[2])_seq$(seq_id).pdf");
            sequence_id=seq_id,
            t_n=t_n
        )
    end
end

println(length(seen_t_ns), " unique t_n combinations plotted.")
println(length(seen_seq_ids), " unique sequence_ids plotted.")

Progress: 100%|█████████████████████████████████████████| Time: 0:07:39


101 unique t_n combinations plotted.
1 unique sequence_ids plotted.


In [ ]:
## Make eigs vs. rho + global MP plot for fixed mu

## calculate rho for all rows
df[!, :rho] = [row.t_n[2] / row.t_n[1] for row in eachrow(df)]

dir = joinpath(results_dir, "eigs_vs_rho_with_mp/$unique_path")
isdir(dir) || mkpath(dir)

restricted_mu_range = mu_range

for mu in restricted_mu_range
    for seq_id in sequence_ids
        plt_eigenvalues_with_mp(
            df,
            :rho,
            joinpath(dir, "eigenvalues_vs_rho_with_mp_N$(N_range[1])_Delta$(Delta_range[1])_mu$(mu)_seq$(seq_id).pdf");
            sequence_id=seq_id,
            mu=mu
        )
    end
end

In [ ]:
## Make local wavefunction plots

println(Delta_range[1])
println(N_range[1])

dir = joinpath(results_dir, "local_wavefunctions/$unique_path")
isdir(dir) || mkpath(dir)

# # every step= out of the full mu_range
# restricted_step = 0.1
# restricted_mu_range = collect(-3.0:restricted_step:3.0)
# restricted_mu_range = filter(x -> isapprox(mod(x, restricted_step), 0.0; atol=1e-8), mu_range)
restricted_mu_range = mu_range

@showprogress for seq_id in sequence_ids
    subdir = joinpath(dir, "seq_$(seq_id)")
    isdir(subdir) || mkdir(subdir)
    for mu in restricted_mu_range
        for t_n in t_combinations
            plt_local_mbs_wavefunction(
                df,
                joinpath(subdir, "local_wavefunction_N$(N_range[1])_Delta$(Delta_range[1])_t1$(t_n[1])_t2$(t_n[2])_mu$(mu)_seq$(seq_id).svg");
                sequence_id=seq_id,
                mu=mu,
                t_n=t_n
            )
        end
    end
end

0.05
100


In [65]:
## Calculate global and local MP for all scales

## Global MP
mp_col = Vector{Float64}(undef, nrow(df))
@showprogress for (i, row) in enumerate(eachrow(df))
    mp_col[i] = np_calc_maj_mp(row.eigenvectors)
end
df[!, :mp] = mp_col

## Local MP for all scales
local_mp_col = Vector{Vector{NamedTuple}}(undef, nrow(df))
@showprogress for (i, row) in enumerate(eachrow(df))
    local_mp_col[i] = calc_local_scales_mp_allscales_optimised(row.eigenvectors)
end
df[!, :local_mp] = local_mp_col

println("Calculated global and local MP for all scales.")

Progress: 100%|█████████████████████████████████████████| Time: 0:00:00


Calculated global and local MP for all scales.


In [66]:
## Plot local MP across scales

dir = joinpath(results_dir, "local_mp_scales/$unique_path/pdf")
isdir(dir) || mkpath(dir)

# restricted_mu_range = [0.0]

# subdir = joinpath(dir, "mu0.0")
# isdir(subdir) || mkdir(subdir)
# @showprogress for seq_id in sequence_ids
#     for mu in restricted_mu_range
#         for t_n in t_combinations
#             for Delta in Delta_range
#                 plt_local_mp_scales_middle_site(
#                     df,
#                     joinpath(subdir, "local_mp_scales_seq_$(seq_id)_N$(N_range[1])_Delta$(Delta)_t1$(t_n[1])_t2$(t_n[2])_mu$(mu).pdf");
#                     sequence_id=seq_id,
#                     mu=mu,
#                     t_n=t_n
#                 )
#             end
#         end
#     end
# end

restricted_mu_range = mu_range

@showprogress for seq_id in sequence_ids
    for mu in restricted_mu_range
        subdir = joinpath(dir, "mu$(mu)_seq_$(seq_id)")
        isdir(subdir) || mkdir(subdir)
        for t_n in t_combinations
            for Delta in Delta_range
                plt_local_mp_scales_middle_site(
                    df,
                    joinpath(subdir, "local_mp_scales_mu_$(mu)_N$(N_range[1])_Delta$(Delta)_t1$(t_n[1])_t2$(t_n[2])_seq$(seq_id).pdf");
                    log=false,
                    sequence_id=seq_id,
                    mu=mu,
                    t_n=t_n
                )
            end
        end
    end
end

In [ ]:
## Calculate Spectral Function

eta = 0.1
omega_range = collect(-3.0:0.01:3.0)

spectral_df = calc_spectral_func_for_all!(
    df, eta, omega_range)
println("DataFrame with spectral functions computed:")

Progress:  36%|██████████████▉                          |  ETA: 6:48:33m

In [ ]:
## Global MP
mp_col = Vector{Float64}(undef, nrow(spectral_df))
@showprogress for (i, row) in enumerate(eachrow(spectral_df))
    mp_col[i] = np_calc_maj_mp(row.eigenvectors)
end
df[!, :mp] = mp_col

println("Calculated global MP for spectral function DataFrame.")

Calculated global MP for spectral function DataFrame.


In [ ]:
## Plot spectral function

dir = joinpath(results_dir, "spectral_functions/$unique_path/pdf")
isdir(dir) || mkpath(dir)

restricted_mu_range = mu_range

@showprogress for seq_id in sequence_ids
    subdir = joinpath(dir, "seq_$(seq_id)_w$(omega_range[1])-$(omega_range[end])")
    isdir(subdir) || mkdir(subdir)
    println("Plotting spectral functions for sequence $(seq_id) in subdir: $(subdir)")
    for mu in restricted_mu_range
        for t_n in t_combinations
            for Delta in Delta_range
                plt_spectral_function(
                    spectral_df,
                    omega_range,
                    joinpath(subdir, "spectral_function_seq_with_mp_$(seq_id)_N$(N_range[1])_Delta$(Delta)_t1$(t_n[1])_t2$(t_n[2])_mu$(mu).pdf");
                    sequence_id=seq_id,
                    mu=mu,
                    t_n=t_n,
                    Delta=Delta
                )
            end
        end
    end
end

Plotting spectral functions for sequence (0.6180048661800487, 0.0) in subdir: /Users/Will/Documents/Quasicrystal_Majorana_project_clone/Quasicrystal_Majorana_project/working_notebooks/PRB_test_results/spectral_functions/N(500-500-1)_mu(0.000-2.000-21)_t11.000-1.000-1_t20.500-0.500-1_Delta0.125-0.125-1_slope(0.618-0.618-1)_phason(0.000-0.000-1)/pdf/seq_(0.6180048661800487, 0.0)_w-3.0-3.0


### 2.0 Fixed $\rho/\Delta$

In [ ]:
## Solve for all slopes over range of mu at fixed rho and Delta
## Initialise parameter dataframe 

N_range = [100]
mu_range = [0.1] #collect(range(0.0, stop=3.0, length=31))
t1 = 1.0
rho_Delta = 40.0
t2_range = collect(range(1.0, 50.0, 51))
t_combinations = Vector{Vector{Float64}}()
Delta_range = Float64[]

for t2 in t2_range
    Delta = (t2 / t1) / rho_Delta
    push!(t_combinations, [t1, t2])
    push!(Delta_range, Delta)
end

sequences = seqs
sequence_ids = [(phis[i], phasons[i]) for i in 1:length(seqs)]

fixed_ratio_param_df = initialise_df(
    N_range,
    t_combinations,
    mu_range,
    Delta_range,
    seqs,
    sequence_ids = sequence_ids
)

println("Parameter DataFrame initialised with $(nrow(fixed_ratio_param_df)) rows.")


Parameter DataFrame initialised with 2601 rows.


In [ ]:
println("unique t_ns: ", unique(fixed_ratio_param_df.t_n))
println("unique Deltas: ", unique(fixed_ratio_param_df.Delta))

unique t_ns: [[1.0, 1.0], [1.0, 1.98], [1.0, 2.96], [1.0, 3.94], [1.0, 4.92], [1.0, 5.9], [1.0, 6.88], [1.0, 7.86], [1.0, 8.84], [1.0, 9.82], [1.0, 10.8], [1.0, 11.78], [1.0, 12.76], [1.0, 13.74], [1.0, 14.72], [1.0, 15.7], [1.0, 16.68], [1.0, 17.66], [1.0, 18.64], [1.0, 19.62], [1.0, 20.6], [1.0, 21.58], [1.0, 22.56], [1.0, 23.54], [1.0, 24.52], [1.0, 25.5], [1.0, 26.48], [1.0, 27.46], [1.0, 28.44], [1.0, 29.42], [1.0, 30.4], [1.0, 31.38], [1.0, 32.36], [1.0, 33.34], [1.0, 34.32], [1.0, 35.3], [1.0, 36.28], [1.0, 37.26], [1.0, 38.24], [1.0, 39.22], [1.0, 40.2], [1.0, 41.18], [1.0, 42.16], [1.0, 43.14], [1.0, 44.12], [1.0, 45.1], [1.0, 46.08], [1.0, 47.06], [1.0, 48.04], [1.0, 49.02], [1.0, 50.0]]
unique Deltas: [0.025, 0.0495, 0.074, 0.0985, 0.123, 0.14750000000000002, 0.172, 0.1965, 0.221, 0.2455, 0.27, 0.2945, 0.319, 0.3435, 0.368, 0.39249999999999996, 0.417, 0.4415, 0.466, 0.49050000000000005, 0.515, 0.5395, 0.564, 0.5885, 0.613, 0.6375, 0.662, 0.6865, 0.7110000000000001, 0.7355, 0

In [ ]:
fixed_ratio_df = solve_for_all!(fixed_ratio_param_df)
println("solved with $(nrow(fixed_ratio_df)) rows.")

Progress:  99%|████████████████████████████████████████▌|  ETA: 0:00:00

LoadError: InterruptException:

In [ ]:
## Calculate global and local MP for all scales

## Global MP
mp_col = Vector{Float64}(undef, nrow(fixed_ratio_df))
@showprogress for (i, row) in enumerate(eachrow(fixed_ratio_df))
    mp_col[i] = np_calc_maj_mp(row.eigenvectors)
end
fixed_ratio_df[!, :mp] = mp_col

## Local MP for all scales
local_mp_col = Vector{Vector{NamedTuple}}(undef, nrow(fixed_ratio_df))
@showprogress for (i, row) in enumerate(eachrow(fixed_ratio_df))
    local_mp_col[i] = calc_local_scales_mp(row.eigenvectors)
end
fixed_ratio_df[!, :local_mp] = local_mp_col

println("Calculated global and local MP for all scales.")

Progress: 100%|█████████████████████████████████████████| Time: 0:00:00


Calculated global and local MP for all scales.


In [ ]:
dir = joinpath(results_dir, "local_mp_scales/mu$(mu_range[1])_rho_Delta_ratio$(rho_Delta)/pdf")
isdir(dir) || mkpath(dir)

restricted_mu_range = mu_range

for row in eachrow(fixed_ratio_df)
    Delta = row.Delta
    t_n = row.t_n
    mu=row.mu
    seq_id = row.sequence_id

    rho_Delta_ratio = (t_n[2]/t_n[1]) / Delta
    if isapprox(rho_Delta_ratio, rho_Delta; atol=1e-9)
        savepath = "mu$(mu)_seq_$(seq_id)_Delta$(Delta)_t1$(t_n[1])_t2$(t_n[2]).pdf"
        plt_local_mp_scales_middle_site(
            fixed_ratio_df,
            joinpath(dir, savepath);
        )
    end
end

### 2.1 Range of fixed $\rho/\Delta$

In [ ]:
N_range = [200]
mu_range = [0.0, 0.2, 1.0, 1.45]#collect(range(0.0, stop=2.0, length=21))
t1 = 1.0
rho_Delta_range = vcat(collect(range(0.1, 0.9, 9)), collect(range(1.0, 9.0, 9)), collect(range(10.0, 50.0, 5)))

@showprogress for rho_Delta in rho_Delta_range
    t2_range = collect(range(0.0, 1.0, 11))
    t_combinations = Vector{Vector{Float64}}()
    Delta_range = Float64[]

    for t2 in t2_range
        Delta = (t2 / t1) / rho_Delta
        push!(t_combinations, [t1, t2])
        push!(Delta_range, Delta)
    end

    sequences = seqs
    sequence_ids = [(phis[i], phasons[i]) for i in 1:length(seqs)]

    fixed_ratio_param_df = initialise_df(
        N_range,
        t_combinations,
        mu_range,
        Delta_range,
        seqs,
        sequence_ids = sequence_ids
    )

    fixed_ratio_df = solve_for_all_no_prog!(fixed_ratio_param_df)

    ## Global MP
    mp_col = Vector{Float64}(undef, nrow(fixed_ratio_df))
    for (i, row) in enumerate(eachrow(fixed_ratio_df))
        mp_col[i] = np_calc_maj_mp(row.eigenvectors)
    end
    fixed_ratio_df[!, :mp] = mp_col

    ## Local MP for all scales
    local_mp_col = Vector{Vector{NamedTuple}}(undef, nrow(fixed_ratio_df))
    for (i, row) in enumerate(eachrow(fixed_ratio_df))
        local_mp_col[i] = calc_local_scales_mp(row.eigenvectors)
    end
    fixed_ratio_df[!, :local_mp] = local_mp_col

    dir = joinpath(results_dir, "local_mp_scales/rho_Delta_ranges/N$(N_range[1])_mu$(mu_range[1])-$(mu_range[end])-$(length(mu_range))_rho_Delta_ratio$(rho_Delta)")
    isdir(dir) || mkpath(dir)

    restricted_mu_range = mu_range

    for row in eachrow(fixed_ratio_df)
        Delta = row.Delta
        t_n = row.t_n
        mu=row.mu
        seq_id = row.sequence_id

        rho_Delta_ratio = (t_n[2]/t_n[1]) / Delta
        if isapprox(rho_Delta_ratio, rho_Delta; atol=1e-9)
            name = "mu$(mu)_seq_$(seq_id)_Delta$(Delta)_t1$(t_n[1])_t2$(t_n[2]).pdf"
            subdir = joinpath(dir, "Delta$(Delta)_t1$(t_n[1])_t2$(t_n[2])")
            isdir(subdir) || mkdir(subdir)
            plt_local_mp_scales_middle_site(
                fixed_ratio_df,
                joinpath(subdir, name);
                sequence_id = seq_id,
                mu = mu,
                t_n = t_n,
                Delta = Delta
            )
        end
    end
end


Progress: 100%|█████████████████████████████████████████| Time: 0:09:36


In [ ]:
function check_eigenvectors(df::DataFrame; nrows_to_check=5)
    println("Checking eigenvectors for the first $nrows_to_check rows:")
    for (i, row) in enumerate(eachrow(df))
        if i > nrows_to_check
            break
        end
        eigvecs = row.eigenvectors
        if isnothing(eigvecs)
            println("Row $i: eigenvectors is nothing")
            continue
        end
        if any(isnan, eigvecs)
            println("Row $i: eigenvectors contain NaN")
        elseif any(isinf, eigvecs)
            println("Row $i: eigenvectors contain Inf")
        elseif all(x -> abs(x) < 1e-12, eigvecs)
            println("Row $i: all eigenvector entries are (close to) zero")
        else
            minval = minimum(abs, eigvecs)
            maxval = maximum(abs, eigvecs)
            println("Row $i: eigenvectors OK (min abs: $minval, max abs: $maxval)")
        end
    end
end

check_eigenvectors (generic function with 1 method)

In [ ]:
check_eigenvectors(fixed_ratio_df; nrows_to_check=10)

Checking eigenvectors for the first 10 rows:
Row 1: eigenvectors OK (min abs: 0.0, max abs: 1.0)
Row 2: eigenvectors OK (min abs: 0.0, max abs: 0.5000000042215913)
Row 3: eigenvectors OK (min abs: 0.0, max abs: 0.49999992658552667)
Row 4: eigenvectors OK (min abs: 0.0, max abs: 0.4999996401258236)
Row 5: eigenvectors OK (min abs: 0.0, max abs: 0.49999890522137486)
Row 6: eigenvectors OK (min abs: 0.0, max abs: 0.49999742681469334)
Row 7: eigenvectors OK (min abs: 0.0, max abs: 0.49999486130941195)
Row 8: eigenvectors OK (min abs: 0.0, max abs: 0.49999082834517333)
Row 9: eigenvectors OK (min abs: 0.0, max abs: 0.49998492122137433)
Row 10: eigenvectors OK (min abs: 0.0, max abs: 0.4999767158367181)


In [ ]:
function check_global_mp(df::DataFrame; nrows_to_check=5)
    println("Checking global MP for the first $nrows_to_check rows:")
    for (i, row) in enumerate(eachrow(df))
        if i > nrows_to_check
            break
        end
        mp = getproperty(row, :mp)
        if isnothing(mp)
            println("Row $i: mp is nothing")
        elseif isnan(mp)
            println("Row $i: mp is NaN")
        elseif isinf(mp)
            println("Row $i: mp is Inf")
        elseif abs(mp) < 1e-12
            println("Row $i: mp is (close to) zero")
        elseif abs(mp) > 2
            println("Row $i: mp is suspiciously large: $mp")
        else
            println("Row $i: mp = $mp")
        end
    end
end

function check_local_mp(df::DataFrame; nrows_to_check=3, nlocal=5)
    println("Checking local MP for the first $nrows_to_check rows (showing first $nlocal entries per row):")
    for (i, row) in enumerate(eachrow(df))
        if i > nrows_to_check
            break
        end
        local_mp = getproperty(row, :local_mp)
        if isnothing(local_mp)
            println("Row $i: local_mp is nothing")
            continue
        end
        if isempty(local_mp)
            println("Row $i: local_mp is empty")
            continue
        end
        for (j, entry) in enumerate(local_mp[1:min(nlocal, end)])
            mpval = entry[:overall_MP]
            if isnan(mpval)
                println("Row $i, local_mp[$j]: overall_MP is NaN")
            elseif isinf(mpval)
                println("Row $i, local_mp[$j]: overall_MP is Inf")
            elseif abs(mpval) < 1e-12
                println("Row $i, local_mp[$j]: overall_MP is (close to) zero")
            elseif abs(mpval) > 2
                println("Row $i, local_mp[$j]: overall_MP is suspiciously large: $mpval")
            else
                println("Row $i, local_mp[$j]: overall_MP = $mpval")
            end
        end
    end
end

check_local_mp (generic function with 1 method)

In [ ]:
check_global_mp(fixed_ratio_df; nrows_to_check=10)

Checking global MP for the first 10 rows:
Row 1: mp is NaN
Row 2: mp = -5.240825126768859e-5
Row 3: mp = -0.00033656106184020486
Row 4: mp = -0.0009851698756578578
Row 5: mp = -0.002279035171650068
Row 6: mp = -0.004640846056729147
Row 7: mp = -0.00872744349388152
Row 8: mp = -0.015556016536911597
Row 9: mp = -0.02669981710746706
Row 10: mp = -0.04458445570181151


In [ ]:
check_local_mp(fixed_ratio_df; nrows_to_check=3, nlocal=5)

Checking local MP for the first 3 rows (showing first 5 entries per row):
Row 1, local_mp[1]: overall_MP is (close to) zero
Row 1, local_mp[2]: overall_MP is (close to) zero
Row 1, local_mp[3]: overall_MP is (close to) zero
Row 1, local_mp[4]: overall_MP is (close to) zero
Row 1, local_mp[5]: overall_MP is (close to) zero
Row 2, local_mp[1]: overall_MP = -0.7804045742018388
Row 2, local_mp[2]: overall_MP = -0.7857434148884663
Row 2, local_mp[3]: overall_MP = -0.32041104395998404
Row 2, local_mp[4]: overall_MP = -0.23734660822072323
Row 2, local_mp[5]: overall_MP = -0.9960267510023034
Row 3, local_mp[1]: overall_MP = -0.9493601110724037
Row 3, local_mp[2]: overall_MP = -0.9204316193663639
Row 3, local_mp[3]: overall_MP = -0.9999991362278392
Row 3, local_mp[4]: overall_MP = -0.9999999961227911
Row 3, local_mp[5]: overall_MP = -0.999999980811698


In [ ]:


@showprogress for rho_Delta in rho_Delta_range
    ## Global MP
    mp_col = Vector{Float64}(undef, nrow(fixed_ratio_df))
    for (i, row) in enumerate(eachrow(fixed_ratio_df))
        mp_col[i] = np_calc_maj_mp(row.eigenvectors)
    end
    fixed_ratio_df[!, :mp] = mp_col

    ## Local MP for all scales
    local_mp_col = Vector{Vector{NamedTuple}}(undef, nrow(fixed_ratio_df))
    for (i, row) in enumerate(eachrow(fixed_ratio_df))
        local_mp_col[i] = calc_local_scales_mp(row.eigenvectors)
    end
    fixed_ratio_df[!, :local_mp] = local_mp_col

    dir = joinpath(results_dir, "local_mp_scales/rho_Delta_ranges/mu$(mu_range[1])-$(mu_range[end])-$(length(mu_range))_rho_Delta_ratio$(rho_Delta)")
    isdir(dir) || mkpath(dir)

    restricted_mu_range = mu_range

    for row in eachrow(fixed_ratio_df)
        Delta = row.Delta
        t_n = row.t_n
        mu=row.mu
        seq_id = row.sequence_id

        rho_Delta_ratio = (t_n[2]/t_n[1]) / Delta
        if isapprox(rho_Delta_ratio, rho_Delta; atol=1e-9)
            name = "mu$(mu)_seq_$(seq_id)_Delta$(Delta)_t1$(t_n[1])_t2$(t_n[2]).pdf"
            subdir = joinpath(dir, "Delta$(Delta)_t1$(t_n[1])_t2$(t_n[2])")
            isdir(subdir) || mkdir(subdir)
            plt_local_mp_scales_middle_site(
                fixed_ratio_df,
                joinpath(subdir, name);
                sequence_id = seq_id,
                mu = mu,
                t_n = t_n,
                Delta = Delta
            )
        end
    end
end


Progress: 100%|█████████████████████████████████████████| Time: 0:00:00
Progress: 100%|█████████████████████████████████████████| Time: 0:00:00
Progress: 100%|█████████████████████████████████████████| Time: 0:00:13
